# Setup
Reads results produced by the batch analysis script and consolidates them into charts and tables for the thesis.

**Directory layout expected:**
```bash
Results/
  <project-name>/
    shallow/            (depth= 1,  stdlib included)
    deep/               (depth=-1,  stdlib included)
    shallow_no_stdlib/  (depth= 1,  stdlib excluded)
    deep_no_stdlib/     (depth=-1,  stdlib excluded)
      report.json
```

In [ ]:
import json
import re
from pathlib import Path
from collections import defaultdict

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from scipy import stats

sns.set_theme(style="whitegrid", palette="muted", font_scale=1.1)

CASE_ORDER   = ["shallow", "shallow_no_stdlib", "deep", "deep_no_stdlib"]
CASE_LABELS  = {
    "shallow"          : "Depth 1",
    "shallow_no_stdlib": "Depth 1 (no stdlib)",
    "deep"             : "Full depth",
    "deep_no_stdlib"   : "Full depth (no stdlib)",
}

CASE_COLORS = {
    # Royal Blue Family (Depth 1)
    "shallow": "#2A52BE",            # Mid-range balanced Royal Blue
    "shallow_no_stdlib": "#122352",   # Sophisticated Deep Navy/Dark Blue
    
    # Crimson Family (Full Depth)
    "deep": "#B21E35",               # Rich, classic Crimson (not overly bright)
    "deep_no_stdlib": "#5C0F1B",      # Deep, elegant Burgundy/Wine Crimson
}

RESULTS_DIR = Path.home() / "UvA" / "Thesis" / "Results"
print(f"Results directory: {RESULTS_DIR}")
print(f"Exists: {RESULTS_DIR.exists()}")

In [ ]:
import json
from pathlib import Path

# Assuming CASE_ORDER and RESULTS_DIR are defined elsewhere


def load_results(results_dir: Path) -> dict:
    """Returns a nested dict: data[project_name][case_name] = parsed JSON dict"""
    return {
        p.name: {
            c: json.loads(report.read_text(encoding="utf-8"))
            for c in CASE_ORDER
            if (report := next((p / c).rglob("*.json"), None))
        }
        for p in sorted(results_dir.iterdir())
        if p.is_dir()
    }


# 1. Load data
RAW = load_results(RESULTS_DIR)

# 2. Print aligned table using standard f-string formatting
print(f"\n{'PROJECT':<15} {'| LOADED CASES'}")
print("-" * 70)

for proj, cases in RAW.items():
    cases_str = " | ".join(cases.keys()) if cases else "[No Cases Found]"
    print(f"{proj:<15} | {cases_str}")

---
## Statistics

In [ ]:
def compute_package_rows(raw: dict) -> pd.DataFrame:
    """Flattens raw JSON data into a DataFrame with one row per (project, case, package)."""
    rows = [
        {
            "project"       : project , "case" : case, "pkg_path": pkg_path,
            "dep_group"     : pkg_path, "depth": pkg.get("depth", -1),
            
            "found_reached" : (tot      := pkg.get("reachableFunctions", 0)),

            "func_total"    : (tot      := pkg.get("functionCount", 0)),
            "func_unused"   : (un       := len(pkg.get("unusedFunctions", []))),
            "func_reached"  : (reach    := max(tot - un, 0)),

            "acr"           : reach / tot if tot > 0 else None,
            "edge_total"    : pkg.get("edges", {}).get("total", 0),
            "is_stdlib"     : pkg.get("isStdlib", False),
            "is_external"   : pkg.get("depth") > 0,
        }
        for project , cases     in raw.items()
        for case    , report    in cases.items()
        for pkg_path, pkg       in report.get("packages", {}).items()
    ]
    return pd.DataFrame(rows)

PKG_DF = compute_package_rows(RAW)
print(f"Package rows: {len(PKG_DF):,}")
print(PKG_DF.head(2000).to_string())

In [ ]:
def compute_project_rows(raw: dict, pkg_df: pd.DataFrame) -> pd.DataFrame:
    """Flattens raw data into a DataFrame with one row per (project, case)."""
    # 1. Pre-calculate external functions reached per (project, case) in one go
    all_reached_map = (
        pkg_df
        .groupby(["project", "case"])["func_reached"]
        .sum()
        .to_dict()
    )
    ext_reached_map = (
        pkg_df[pkg_df["is_external"]]
        .groupby(["project", "case"])["func_reached"]
        .sum()
        .to_dict()
    )

    # 2. Build rows using dictionary comprehension
    rows = [
        {
            "project"           : proj,
            "case"              : case,
            "total_funcs"       : (tot := rep.get("totalFunctions", 0)),
            "reachable_funcs"   : (reach := rep.get("reachableFunctions", 0)),
            "reachability_ratio": reach / tot if tot > 0 else None,
            "sci"               : ext_reached_map.get((proj, case), 0) / all_reached_map.get((proj, case), 1),
            "static_sites"      : (s_s := (ind := rep.get("indirect", {})).get("staticCallSites", 0)),
            "interface_sites"   : (i_s := ind.get("interfaceCallSites", 0)),
            "funcvar_sites"     : (f_s := ind.get("funcVarCallSites", 0)),
            "total_sites"       : (tot_s := s_s + i_s + f_s),
            "indirect_ratio"    : (i_s + f_s) / tot_s if tot_s > 0 else None,
        }
        for proj, cases in raw.items()
        for case, rep in cases.items()
    ]

    return pd.DataFrame(rows)


PROJ_DF = compute_project_rows(RAW, PKG_DF)
print(f"Project rows: {len(PROJ_DF)}")
print(PROJ_DF.head(2000).to_string())

___
# Utility stuff

In [ ]:
def render_grouped_distribution_plot(
    pivot_df: pd.DataFrame,
    title: str,
    ylabel: str,
    save_filename: str,
    y_formatter=None,
    ylim=None,
    # Placed outside the axes boundaries by default using bbox_to_anchor
    legend_loc="upper left",
    bbox_to_anchor=(1.02, 1.0),
    custom_legend_setup=None,
    use_pandas_plot=False,
    spacing=1.5,
):
    """Encapsulates the standard boilerplate for project-by-case grouped visualizations

    with an external legend configuration by default to avoid masking bar elements.
    """
    n_proj = len(pivot_df.index)

    # Standardized Figure Canvas Scaling
    fig_width = max(11, n_proj * 1.6)
    fig, ax = plt.subplots(figsize=(fig_width, 6))

    # Plot Generation Router
    if use_pandas_plot:
        colors = [CASE_COLORS[c] for c in pivot_df.columns]
        alphas = [0.55 if "no_stdlib" in c else 0.95 for c in pivot_df.columns]
        labels = [CASE_LABELS[c] for c in pivot_df.columns]

        pivot_df.plot(kind="bar", width=0.8, color=colors, ax=ax, zorder=3)

        for container, alpha in zip(ax.containers, alphas):
            plt.setp(container, alpha=alpha)
    else:
        labels = []
        if custom_legend_setup:
            # Pass bbox parameters down to custom plotting loops if they handle their own legends
            labels = custom_legend_setup(
                ax, pivot_df, spacing, legend_loc, bbox_to_anchor
            )

    # Structural Aesthetics and Typographical Layout
    ax.set_title(title, pad=15, fontweight="medium")
    ax.set_ylabel(ylabel, labelpad=10)
    ax.set_xlabel("")

    if not use_pandas_plot:
        ax.set_xticks(np.arange(n_proj) * spacing)
    ax.set_xticklabels(pivot_df.index, rotation=30, ha="right")

    if ylim:
        ax.set_ylim(ylim)
    if y_formatter:
        ax.yaxis.set_major_formatter(y_formatter)

    ax.grid(axis="y", linestyle="--", alpha=0.5, zorder=0)

    # Default Legend Handler (Used for Pandas plots or default custom drop-throughs)
    if use_pandas_plot:
        ax.legend(
            labels,
            loc=legend_loc,
            bbox_to_anchor=bbox_to_anchor,
            framealpha=0.9,
            borderaxespad=0,
        )
    elif not custom_legend_setup:
        ax.legend(
            loc=legend_loc,
            bbox_to_anchor=bbox_to_anchor,
            framealpha=0.9,
            borderaxespad=0,
        )

    # Use bbox_inches="tight" to make sure the external legend isn't clipped in the saved PDF
    plt.savefig(save_filename, bbox_inches="tight")
    plt.show()
    return fig, ax

___
### Q1 & Q3

In [ ]:
def compute_project_generalization(pkg_df: pd.DataFrame) -> pd.DataFrame:
    # 1. Collapse all 13,000 rows straight down to just Project + Case
    proj_grp = (
        pkg_df.groupby(["project", "case"], as_index=False)
        .agg(
            reached=("func_reached", "sum"),
            total=("func_total", "sum")
        )
    )
    
    # 2. Compute the true absolute coverage ratio of the entire project footprint
    proj_grp["project_acr"] = proj_grp["reached"] / proj_grp["total"].replace(0, np.nan)
    
    return proj_grp

GEN_PROJ_DF = compute_project_generalization(PKG_DF)
print(f"Compressed Rows: {len(GEN_PROJ_DF)}") # Will equal (Number of Unique Projects * 4 Cases)
print(GEN_PROJ_DF.head(100).to_string())

___
## Some Plots

In [ ]:
def plot_acr_grouped(proj_df: pd.DataFrame) -> None:
    pivot = (
        proj_df.pivot(index="project", columns="case", values="project_acr")
        .reindex(columns=CASE_ORDER)
        .sort_index()
    )

    render_grouped_distribution_plot(
        pivot_df=pivot,
        title="Project ACR per Analysis Configuration",
        ylabel="Absolute Coverage Ratio (ACR)",
        save_filename="project_acr_distribution.pdf",
        y_formatter=mticker.PercentFormatter(xmax=1),
        ylim=(0, 1.05),
        legend_loc="lower right",
        use_pandas_plot=True,
    )

plot_acr_grouped(GEN_PROJ_DF)

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns


def plot_package_acr_distribution(pkg_df: pd.DataFrame):
    """Renders a clean boxplot distribution showing package-level ACR

    across configurations grouped by project, with a horizontal layout legend up top.
    """
    plot_df = pkg_df[pkg_df["func_total"] > 0].copy()

    n_proj = plot_df["project"].nunique()
    # Boosted height slightly to 6.5 to give the top legend breathing room below the title
    fig, ax = plt.subplots(figsize=(max(12, n_proj * 2.2), 6.5))

    # Render the distribution curves
    sns.boxplot(
        data=plot_df,
        x="project",
        y="acr",
        hue="case",
        hue_order=CASE_ORDER,
        palette=CASE_COLORS,
        ax=ax,
        fliersize=2,
        linewidth=1.5,
        boxprops=dict(edgecolor="black", alpha=0.25),
        whiskerprops=dict(color="black", alpha=1.0),
        capprops=dict(color="black", alpha=1.0),
        medianprops=dict(color="black", linewidth=2.0, alpha=1.0),
    )

    # Styling & Typography
    # Increased title pad to 35 so it sits cleanly above the new top horizontal legend
    ax.set_title(
        "Distribution of Package Absolute Coverage Ratio (ACR) per Configuration",
        pad=35,
        fontweight="medium",
    )
    ax.set_ylabel("Package ACR")
    ax.set_xlabel("")
    ax.set_ylim(-0.05, 1.05)

    import matplotlib.ticker as mticker

    ax.yaxis.set_major_formatter(mticker.PercentFormatter(xmax=1))

    ax.set_xticks(range(n_proj))
    ax.set_xticklabels(
        [t.get_text() for t in ax.get_xticklabels()], rotation=30, ha="right"
    )
    ax.grid(axis="y", linestyle="--", alpha=0.4, zorder=0)

    # Fixed: Positioned centrally above the plot area in a flat horizontal line
    ax.legend(
        title="",  # Kept blank to keep the top bar clean and compact
        loc="lower center",
        bbox_to_anchor=(0.5, 1.01),
        ncol=len(CASE_ORDER),  # Spreads items horizontally
        framealpha=0.9,
        borderaxespad=0,
    )

    plt.tight_layout()
    plt.savefig("package_acr_distribution_boxplot.pdf", bbox_inches="tight")
    plt.show()


# Run the visualization setup
plot_package_acr_distribution(PKG_DF)

In [ ]:
def plot_sci_grouped(proj_df: pd.DataFrame) -> None:
    pivot = (
        proj_df.pivot(index="project", columns="case", values="sci")
        .reindex(columns=CASE_ORDER)
        .sort_index()
    )

    render_grouped_distribution_plot(
        pivot_df=pivot,
        title="RQ1 — SCI per project and analysis configuration",
        ylabel="SCI (external functions reached / reachable functions)",
        save_filename="rq1_sci.pdf",
        y_formatter=mticker.PercentFormatter(xmax=1),
        ylim=(0, 1.05),
        legend_loc="lower right",
        use_pandas_plot=True,
    )
    

plot_sci_grouped(PROJ_DF)

In [ ]:
def plot_combined_total_vs_reached(proj_df: pd.DataFrame):
    pivot = proj_df.pivot(
        index="project", columns="case", values=["total_funcs", "reachable_funcs"]
    )
    pivot = pivot.reindex(
        pivot["total_funcs"].max(axis=1).sort_values(ascending=False).index
    )

    # Fixed: Added 'loc' and 'bbox' parameters to the function signature
    def draw_total_vs_reached_bars(ax, pivot_df, space_step, loc, bbox):
        n_cases = len(CASE_ORDER)
        base_pos = np.arange(len(pivot_df.index)) * space_step
        width = 0.85 / n_cases

        for i, case in enumerate(CASE_ORDER):
            pos = base_pos + (i - (n_cases - 1) / 2) * width
            color = CASE_COLORS[case]

            ax.bar(
                pos,
                pivot_df[("total_funcs", case)],
                width,
                color=color,
                alpha=0.25,
                edgecolor=color,
                zorder=2,
            )
            ax.bar(
                pos,
                pivot_df[("reachable_funcs", case)],
                width,
                color=color,
                alpha=0.95,
                label=CASE_LABELS[case],
                zorder=3,
            )

        handles, labels = ax.get_legend_handles_labels()
        handles.insert(
            0, plt.Rectangle((0, 0), 1, 1, color="green", alpha=0.25, edgecolor="gray")
        )
        labels.insert(0, "Total Functions Scope")
        
        # Fixed: Hooked up loc, bbox_to_anchor, and borderaxespad to respect the outer layout engine
        ax.legend(
            handles, 
            labels, 
            loc=loc, 
            bbox_to_anchor=bbox, 
            framealpha=0.9, 
            borderaxespad=0
        )

    render_grouped_distribution_plot(
        pivot_df=pivot,
        title="RQ1 - Total vs Reachable Functions across Analysis Configurations",
        ylabel="Function Count",
        save_filename="rq1_absolute_combined.pdf",
        y_formatter=mticker.StrMethodFormatter("{x:,.0f}"),
        custom_legend_setup=draw_total_vs_reached_bars,
        spacing=1.0,
    )

# This will now compile cleanly without argument alignment issues
plot_combined_total_vs_reached(PROJ_DF)

________
## Q4


In [ ]:
def compute_q4_indirect_metrics(raw: dict) -> pd.DataFrame:
    """Extracts detailed indirect call and function propagation metrics."""
    rows = []
    for proj, cases in raw.items():
        for case, rep in cases.items():
            ind = rep.get("indirect", {})
            rows.append({
                "project": proj,
                "case": case,
                "staticCallSites"       : ind.get("staticCallSites"     , 0),
                "interfaceCallSites"    : ind.get("interfaceCallSites"  , 0),
                "funcVarCallSites"      : ind.get("funcVarCallSites"    , 0),
                "funcLiteralStores"     : ind.get("funcLiteralStores"   , 0),
                "funcNamedStores"       : ind.get("funcNamedStores"     , 0),
                "funcPropagations"      : ind.get("funcPropagations"    , 0),
                "funcInStructOrMap"     : ind.get("funcInStructOrMap"   , 0),
                "funcChans"             : ind.get("funcChans"           , 0),
                "goroutinesFuncChan"    : ind.get("goroutinesFuncChan"  , 0),
                "funcsSentToFuncChan"   : ind.get("funcsSentToFuncChan" , 0),
                "funcsReceivedForCall"  : ind.get("funcsReceivedForCall", 0)
            })
    return pd.DataFrame(rows)

INDIRECT_DF = compute_q4_indirect_metrics(RAW)
print(f"Indirect Metrics rows: {len(INDIRECT_DF)}")
INDIRECT_DF.head(10)

In [ ]:
def compute_signature_metrics(raw: dict, target_case="deep") -> pd.DataFrame:
    """Extracts signature metrics for potential targets vs actual call sites."""
    rows = []
    for proj, cases in raw.items():
        rep = cases.get(target_case, {})
        sigs = rep.get("indirect", {}).get("signatureMetrics", {})
        for sig, metrics in sigs.items():
            rows.append({
                "project": proj,
                "signature": sig,
                "potentialTargets": metrics.get("potentialTargets", 0),
                "actualCallSites": metrics.get("actualCallSites", 0)
            })
    return pd.DataFrame(rows)

SIG_DF = compute_signature_metrics(RAW, "deep")

# Aggregate by project to calculate the average signature ambiguity
sig_agg = SIG_DF.groupby("project")[["potentialTargets", "actualCallSites"]].sum()
# Ratio: Potential targets that a full pointer analysis would need to disambiguate vs actual call sites
sig_agg["ambiguity_ratio"] = sig_agg["potentialTargets"] / sig_agg["actualCallSites"].replace(0, np.nan)

# Display the aggregated table highlighting projects with the most ambiguity
print("Projects with highest Signature Ambiguity Ratio:")
display(sig_agg.sort_values("ambiguity_ratio", ascending=False))


In [ ]:
def plot_signature_ambiguity_linear(df_agg: pd.DataFrame):
    # Using a linear scale, but perhaps plotting the ratio is cleaner to interpret
    # if you really want raw counts, simply remove set_yscale("log")
    ax = df_agg[["potentialTargets", "actualCallSites"]].sort_values("potentialTargets", ascending=False).plot(
        kind="bar",
        figsize=(12, 6),
        width=0.8,
        color=["#2A52BE", "#B21E35"], 
        alpha=0.9
    )
    
    ax.set_title("RQ4 - Signature Level Ambiguity (Potential Targets vs Actual Indirect Call Sites)")
    ax.set_ylabel("Count")
    ax.set_xlabel("")
    # Removed set_yscale("log") as requested
    ax.set_xticklabels(df_agg.index, rotation=30, ha="right")
    ax.legend(["Potential Targets", "Actual Call Sites"], framealpha=0.9)
    
    plt.tight_layout()
    plt.savefig("rq4_signature_metrics_linear.pdf", bbox_inches="tight")
    plt.show()

plot_signature_ambiguity_linear(sig_agg)

In [ ]:
def plot_indirect_comparison_grouped(df: pd.DataFrame, spacing=1.5):
    # 1. Data Transformation Layer
    df = df.copy()
    cols_to_sum = [
        "funcVarCallSites",
        "funcLiteralStores",
        "funcNamedStores",
        "funcPropagations",
        "funcInStructOrMap",
        "funcChans",
        "goroutinesFuncChan",
        "funcsSentToFuncChan",
        "funcsReceivedForCall",
    ]
    df["Static"], df["Interface"], df["Complex"] = (
        df["staticCallSites"],
        df["interfaceCallSites"],
        df[cols_to_sum].sum(axis=1),
    )

    pivot = df.pivot(
        index="project", columns="case", values=["Static", "Interface", "Complex"]
    )
    sort_order = pivot[("Static", "deep")] + pivot[("Interface", "deep")]
    pivot = pivot.reindex(sort_order.sort_values(ascending=False).index)

    # 2. Spliced Stacked-Bar Logic Block
    # Fixed: Updated function signature to accept loc and bbox from the unified engine
    def draw_stacked_bars(ax, pivot_df, space_step, loc, bbox):
        n_cases = len(CASE_ORDER)
        base_pos = np.arange(len(pivot_df.index)) * space_step
        width = (space_step * 0.7) / n_cases

        for i, case in enumerate(CASE_ORDER):
            pos = base_pos + (i - (n_cases - 1) / 2) * width
            c = CASE_COLORS[case]

            s, j, cp = (
                pivot_df[("Static", case)],
                pivot_df[("Interface", case)],
                pivot_df[("Complex", case)],
            )
            ax.bar(pos, s, width, color=c, alpha=0.9, zorder=3)
            ax.bar(
                pos, j, width, bottom=s, color=c, alpha=0.6, hatch="\\\\\\", zorder=3
            )
            ax.bar(
                pos, cp, width, bottom=s + j, color=c, alpha=0.3, hatch="///", zorder=3
            )

        from matplotlib.patches import Patch

        # Fixed: Tied parameters to outer layout engine constants
        ax.legend(
            [
                Patch(color="gray", alpha=0.9),
                Patch(facecolor="gray", alpha=0.6, hatch="\\\\\\"),
                Patch(facecolor="gray", alpha=0.3, hatch="///"),
            ],
            ["Static Call Sites", "Interface Call Sites", "Complex (Indirect/Other)"],
            loc=loc,
            bbox_to_anchor=bbox,
            framealpha=0.9,
            borderaxespad=0,
        )

    # 3. Dispatched Plotting Configuration
    render_grouped_distribution_plot(
        pivot_df=pivot,
        title="RQ4 - Breakdown of Indirect Call Patterns",
        ylabel="Number of Call Sites",
        save_filename="rq4_grouped_comparison_compact.pdf",
        custom_legend_setup=draw_stacked_bars,
        spacing=spacing,
    )


# Running this now will resolve perfectly!
plot_indirect_comparison_grouped(INDIRECT_DF, spacing=1.5)

## RQ3

In [ ]:
def compute_utilization_index(raw: dict) -> pd.DataFrame:
    rows = [
        {
            "project":    project,
            "case":       case,
            "pkg_path":   pkg_path,
            "func_total": pkg.get("functionCount", 0),
            "unused":     set(pkg.get("unusedFunctions", [])),
            "is_stdlib":  pkg.get("isStdlib", False),
        }
        for project, cases   in raw.items()
        for case,    data    in cases.items()
        for pkg_path, pkg    in data["packages"].items()
        if  pkg.get("depth", 0) > 0
    ]
 
    return (
        pd.DataFrame(rows)
        .groupby(["pkg_path", "case"], sort=False)
        .agg(
            n_deps     = ("project",   "nunique"),
            func_total = ("func_total","first"),
            never_used = ("unused",    lambda x: set.intersection(*x)),
            is_stdlib  = ("is_stdlib", "first"),
        )
        .assign(
            u           = lambda df: df["func_total"] - df["never_used"].str.len(),
            utilization = lambda df: df["u"] / df["func_total"].replace(0, np.nan),
        )
        .reset_index()
        .sort_values(["case", "utilization"], ascending=[True, False])
    )
 
 
# ================================================================================================
 
def plot_utilization_distribution(util_df: pd.DataFrame) -> None:
    plot_df = util_df.dropna(subset=["utilization"])
    cases   = [c for c in CASE_ORDER if c in plot_df["case"].unique()]
 
    fig, ax = plt.subplots(figsize=(10, 5))
 
    for pos, case in enumerate(cases):
        data  = plot_df.loc[plot_df["case"] == case, "utilization"]
        color = CASE_COLORS[case]
 
        vp = ax.violinplot(data, positions=[pos], showmedians=True)
        for part in vp["bodies"]:
            part.set_facecolor(color)
            part.set_alpha(0.55)
        for key in ("cmedians", "cmins", "cmaxes", "cbars"):
            vp[key].set_edgecolor(color)
            vp[key].set_linewidth(1.5)
 
        jitter = np.random.uniform(-0.1, 0.1, len(data))
        ax.scatter(pos + jitter, data, color=color, alpha=0.3, s=10)
        ax.text(pos, data.median() + 0.03, f"{data.median():.2f}",
                ha="center", fontweight="bold", color=color, fontsize=8)
 
    ax.set_xticks(range(len(cases)))
    ax.set_xticklabels([CASE_LABELS.get(c, c) for c in cases], rotation=15, ha="right")
    ax.set(title="Utilization Index Distribution", ylim=(-0.05, 1.1),
           ylabel="U = u / func_total")
    ax.axhline(0.5, color="grey", linestyle="--", alpha=0.5, label="U = 0.5")
    ax.legend(fontsize=8)
    ax.grid(axis="y", linestyle="--", alpha=0.3)
    plt.tight_layout()
    plt.savefig("utilization_index_distribution.pdf", bbox_inches="tight")
    plt.show()
 
 
# ================================================================================================
 
def print_top_bottom(util_df: pd.DataFrame, case: str, n: int = 10) -> None:
    sub = (
        util_df[(util_df["case"] == case) & (util_df["n_deps"] > 1)]
        .dropna(subset=["utilization"])
        .sort_values("utilization", ascending=False)
        [["pkg_path", "n_deps", "func_total", "u", "utilization"]]
    )
    for label, df in [("Top", sub.head(n)), ("Bottom", sub.tail(n).iloc[::-1])]:
        print(f"\n── {label} {n} [{CASE_LABELS[case]}] ──")
        print(df.to_string(index=False))
 


# ================================================================================================
UTIL_DF = compute_utilization_index(RAW)
 
print(f"Utilization rows: {len(UTIL_DF):,}")
display(
    UTIL_DF.groupby("case")["utilization"]
    .agg(["count", "mean", "median", "std"])
    .reindex(CASE_ORDER)
    .rename(columns={"count": "n_pkgs", "mean": "mean_U", "median": "median_U", "std": "std_U"})
    .round(4)
)
 
plot_utilization_distribution(UTIL_DF)
 
for case in ["deep_no_stdlib", "shallow_no_stdlib"]:
    print_top_bottom(UTIL_DF, case)
